# workflow
1. Inspect saved model files and training outputs.
2. Check class balance and label encoding.
3. Check predicted probabilities and decision thresholds.
4. Identify likely causes of model bias.
5. Save a debugging report for your project.

In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE = Path("D:/adaptive_federated_gnn")
OUT = BASE / "outputs" / "day24"
OUT.mkdir(parents=True, exist_ok=True)

folders_to_check = [
    BASE / "outputs" / "day16",
    BASE / "outputs" / "day17",
    BASE / "outputs" / "day19",
    BASE / "outputs" / "day20",
]

for folder in folders_to_check:
    print(f"\n{'=' * 55}")
    print("Folder:", folder)
    print("Exists:", folder.exists())

    if folder.exists():
        for path in sorted(folder.iterdir()):
            print(" ", path.name)



Folder: D:\adaptive_federated_gnn\outputs\day16
Exists: True
  client_1_local_model.pt
  client_2_local_model.pt
  client_3_local_model.pt
  fedavg_client_weights.csv
  local_training_metrics.csv
  non_iid_classification_report.csv
  non_iid_client_distribution.csv
  non_iid_clients.pt
  non_iid_confusion_matrix.csv
  non_iid_distribution.png
  non_iid_global_model.pt
  non_iid_global_test_metrics.csv
  non_iid_test_predictions.csv

Folder: D:\adaptive_federated_gnn\outputs\day17
Exists: True
  client_1_retrained_model.pt
  client_2_retrained_model.pt
  client_3_retrained_model.pt
  privacy_update_diagnostics.csv
  private_fedavg_weights.csv
  private_global_model.pt
  private_model_test_metrics.csv
  private_model_test_predictions.csv
  protected_client_updates.pt
  shared_initial_model.pt

Folder: D:\adaptive_federated_gnn\outputs\day19
Exists: True
  adaptive_client_weights.csv
  adaptive_fedavg_model.pt
  adaptive_fedavg_predictions.csv
  client_weight_comparison.png
  fedavg_comp

In [2]:

predictions_path = (
    BASE / "outputs" / "day20" / "three_model_predictions.csv"
)

predictions = pd.read_csv(predictions_path)

print("Total evaluation records:", len(predictions))
print("\nActual label counts:")
display(
    predictions["actual_label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Benign (0)", 1: "Attack (1)"})
    .to_frame("Count")
)

print("\nActual label percentages:")
display(
    (
        predictions["actual_label"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).rename(index={0: "Benign (0)", 1: "Attack (1)"})
     .round(2)
     .to_frame("Percentage")
)

print("\nUnique actual labels:",
      sorted(predictions["actual_label"].dropna().unique()))


Total evaluation records: 85829

Actual label counts:


,Count
actual_label,
Benign (0),38188
Attack (1),47641



Actual label percentages:


,Percentage
actual_label,
Benign (0),44.49
Attack (1),55.51



Unique actual labels: [np.int64(0), np.int64(1)]


In [3]:

prediction_columns = {
    "Standard FedAvg": "standard_fedavg_prediction",
    "Adaptive FedAvg": "adaptive_fedavg_prediction",
    "Private FedAvg": "private_fedavg_prediction",
}

distribution_rows = []

for model_name, column in prediction_columns.items():
    counts = predictions[column].value_counts()

    benign_count = int(counts.get(0, 0))
    attack_count = int(counts.get(1, 0))
    total = len(predictions)

    distribution_rows.append({
        "Model": model_name,
        "Predicted benign": benign_count,
        "Predicted attack": attack_count,
        "Benign (%)": round(100 * benign_count / total, 2),
        "Attack (%)": round(100 * attack_count / total, 2),
    })

distribution_df = pd.DataFrame(distribution_rows)
display(distribution_df)

distribution_df.to_csv(
    OUT / "prediction_distribution.csv",
    index=False
)

print("Saved:", OUT / "prediction_distribution.csv")


,Model,Predicted benign,Predicted attack,Benign (%),Attack (%)
0,Standard FedAvg,83068,2761,96.78,3.22
1,Adaptive FedAvg,83026,2803,96.73,3.27
2,Private FedAvg,1571,84258,1.83,98.17


Saved: D:\adaptive_federated_gnn\outputs\day24\prediction_distribution.csv


In [4]:

probability_columns = {
    "Standard FedAvg": "standard_fedavg_attack_probability",
    "Adaptive FedAvg": "adaptive_fedavg_attack_probability",
    "Private FedAvg": "private_fedavg_attack_probability",
}

probability_rows = []

for model_name, column in probability_columns.items():
    p = pd.to_numeric(
        predictions[column], errors="coerce"
    ).dropna()

    probability_rows.append({
        "Model": model_name,
        "Minimum": p.min(),
        "25th percentile": p.quantile(0.25),
        "Median": p.median(),
        "75th percentile": p.quantile(0.75),
        "Maximum": p.max(),
        "Below 0.5 (%)": 100 * (p < 0.5).mean(),
        "At least 0.5 (%)": 100 * (p >= 0.5).mean(),
    })

probability_df = pd.DataFrame(probability_rows).round(4)
display(probability_df)

probability_df.to_csv(
    OUT / "probability_diagnostics.csv",
    index=False
)

print("Saved:", OUT / "probability_diagnostics.csv")


,Model,Minimum,25th percentile,Median,75th percentile,Maximum,Below 0.5 (%),At least 0.5 (%)
0,Standard FedAvg,0.0,0.4416,0.4797,0.4939,0.8094,96.7831,3.2169
1,Adaptive FedAvg,0.0,0.4422,0.4803,0.4945,0.8126,96.7342,3.2658
2,Private FedAvg,0.0,0.9803,0.9881,0.9922,1.0000,1.8304,98.1696


Saved: D:\adaptive_federated_gnn\outputs\day24\probability_diagnostics.csv


In [5]:

report_path = OUT / "day24_diagnostic_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("DAY 24 — FEDERATED GNN DEBUGGING REPORT\n\n")
    f.write(f"Evaluation records: {len(predictions)}\n\n")

    f.write("PREDICTED CLASS DISTRIBUTIONS\n")
    f.write(distribution_df.to_string(index=False))
    f.write("\n\nPROBABILITY DIAGNOSTICS\n")
    f.write(probability_df.to_string(index=False))
    f.write("\n\nNEXT INVESTIGATION\n")
    f.write(
        "Inspect training labels, output-layer class order, loss function, "
        "local client label distributions, and privacy-update implementation "
        "before retraining or modifying decision thresholds.\n"
    )

print(report_path)
print(report_path.read_text(encoding="utf-8"))


D:\adaptive_federated_gnn\outputs\day24\day24_diagnostic_report.txt
DAY 24 — FEDERATED GNN DEBUGGING REPORT

Evaluation records: 85829

PREDICTED CLASS DISTRIBUTIONS
          Model  Predicted benign  Predicted attack  Benign (%)  Attack (%)
Standard FedAvg             83068              2761       96.78        3.22
Adaptive FedAvg             83026              2803       96.73        3.27
 Private FedAvg              1571             84258        1.83       98.17

PROBABILITY DIAGNOSTICS
          Model  Minimum  25th percentile  Median  75th percentile  Maximum  Below 0.5 (%)  At least 0.5 (%)
Standard FedAvg      0.0           0.4416  0.4797           0.4939   0.8094        96.7831            3.2169
Adaptive FedAvg      0.0           0.4422  0.4803           0.4945   0.8126        96.7342            3.2658
 Private FedAvg      0.0           0.9803  0.9881           0.9922   1.0000         1.8304           98.1696

NEXT INVESTIGATION
Inspect training labels, output-layer class order

In [6]:

from pathlib import Path
import json

BASE = Path("D:/adaptive_federated_gnn")
notebooks_dir = BASE / "notebooks"

search_terms = [
    "CrossEntropyLoss",
    "NLLLoss",
    "optimizer",
    "loss.backward",
    "edge_label",
    "argmax",
    "softmax",
    "private_global_model",
    "noise",
    "clip",
    "load_state_dict",
    "fedavg"
]

target_notebooks = [
    "day11_local_training.ipynb",
    "day12_federated_averaging.ipynb",
    "day16_non_iid_fedavg.ipynb",
    "day17_privacy_mechanism.ipynb",
    "day19_adaptive_client_weighting.ipynb",
]

for name in target_notebooks:
    path = notebooks_dir / name
    print(f"\n{'=' * 70}\nNOTEBOOK: {name}")

    if not path.exists():
        print("Notebook not found")
        continue

    notebook = json.loads(path.read_text(encoding="utf-8"))

    for i, cell in enumerate(notebook.get("cells", []), start=1):
        source = "".join(cell.get("source", []))

        matches = [
            term for term in search_terms
            if term.lower() in source.lower()
        ]

        if matches:
            print(f"\n--- Cell {i} | Matches: {', '.join(matches)} ---")
            print(source[:2200])



NOTEBOOK: day11_local_training.ipynb

--- Cell 10 | Matches: edge_label ---
# Inspect and prepare federated client data

clients = torch.load(
    CLIENT_FILE,
    map_location="cpu",
    weights_only=False
)

print("Client names:", list(clients.keys()))

for client_name, client_data in clients.items():

    print("\nClient:", client_name)

    local_edge_indices = client_data["local_edge_indices"].long()

    client_edge_index = (
        client_data["edge_index"][:, local_edge_indices]
    )

    client_edge_features = client_data["edge_features"]
    client_edge_labels = client_data["edge_labels"]

    print("Client edge index shape:", client_edge_index.shape)
    print("Client edge features shape:", client_edge_features.shape)
    print("Client edge labels shape:", client_edge_labels.shape)

    assert client_edge_index.shape[1] == client_edge_features.shape[0]
    assert client_edge_features.shape[0] == client_edge_labels.shape[0]

print("\nAll client data prepared successfully."

In [7]:

from pathlib import Path
import json

BASE = Path("D:/adaptive_federated_gnn")
NOTEBOOKS = BASE / "notebooks"

targets = {
    "day11_local_training.ipynb": [
        "CrossEntropyLoss", "loss.backward",
        "optimizer.step", "def train", "def forward"
    ],
    "day12_federated_averaging.ipynb": [
        "load_state_dict", "state_dict",
        "average", "fedavg", "weights"
    ],
    "day17_privacy_mechanism.ipynb": [
        "noise", "clip", "norm",
        "state_dict", "load_state_dict"
    ]
}

for filename, terms in targets.items():
    path = NOTEBOOKS / filename
    print(f"\n{'=' * 70}\n{filename}")

    if not path.exists():
        print("Notebook not found")
        continue

    notebook = json.loads(path.read_text(encoding="utf-8"))

    for i, cell in enumerate(notebook.get("cells", []), start=1):
        source = "".join(cell.get("source", []))
        if cell.get("cell_type") != "code":
            continue

        if any(term.lower() in source.lower() for term in terms):
            print(f"\n--- CELL {i} ---")
            print(source[:5000])



day11_local_training.ipynb

--- CELL 12 ---
# Cell 11 — Edge-level GCN model

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class EdgeGCN(nn.Module):

    def __init__(
        self,
        num_nodes,
        edge_feature_size,
        embedding_size=16,
        hidden_size=32
    ):
        super().__init__()

        self.node_embeddings = nn.Embedding(
            num_nodes,
            embedding_size
        )

        self.conv1 = GCNConv(
            embedding_size,
            hidden_size
        )

        self.conv2 = GCNConv(
            hidden_size,
            hidden_size
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                hidden_size * 2 + edge_feature_size,
                64
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, edge_index, edge_features):

        node_ids = torch.arange(
       

In [8]:

import json
from pathlib import Path

path = Path(
    "D:/adaptive_federated_gnn/notebooks/day17_privacy_mechanism.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"], start=1):
    source = "".join(cell.get("source", []))

    terms = [
        "noise_multiplier",
        "clip_norm",
        "add_noise",
        "torch.randn_like",
        "global_state",
        "private_global_model",
        "protected_client_updates",
        "client_update",
        "update_norm",
    ]

    if cell.get("cell_type") == "code" and any(
        term in source.lower() for term in terms
    ):
        print(f"\n--- Day 17, Cell {i} ---")
        print(source[:2200])



--- Day 17, Cell 6 ---
CLIP_NORM = 1.0
NOISE_MULTIPLIER = 0.5

def protect_client_update(local_state, base_state,
                          clip_norm=1.0, noise_multiplier=0.5):
    # Compute one global L2 norm over all parameter updates.
    deltas = {
        key: local_state[key].float() - base_state[key].float()
        for key in base_state
    }

    squared_norm = sum(
        torch.sum(delta ** 2) for delta in deltas.values()
    )
    update_norm = torch.sqrt(squared_norm)

    scale = min(
        1.0,
        clip_norm / (update_norm.item() + 1e-12)
    )

    protected = {}
    for key, delta in deltas.items():
        clipped_delta = delta * scale
        noise = torch.randn_like(clipped_delta) * (
            noise_multiplier * clip_norm
        )
        protected[key] = clipped_delta + noise

    return protected, update_norm.item(), scale

protected_updates = {}
privacy_rows = []

for name, state in retrained_states.items():
    update, original_norm, clip_scale = pro

In [9]:

import json
from pathlib import Path

path = Path(
    "D:/adaptive_federated_gnn/notebooks/day17_privacy_mechanism.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"], start=1):
    source = "".join(cell.get("source", []))

    if "def protect_client_update" in source:
        print(f"Found privacy function in Cell {i}:\n")
        print(source)


Found privacy function in Cell 6:

CLIP_NORM = 1.0
NOISE_MULTIPLIER = 0.5

def protect_client_update(local_state, base_state,
                          clip_norm=1.0, noise_multiplier=0.5):
    # Compute one global L2 norm over all parameter updates.
    deltas = {
        key: local_state[key].float() - base_state[key].float()
        for key in base_state
    }

    squared_norm = sum(
        torch.sum(delta ** 2) for delta in deltas.values()
    )
    update_norm = torch.sqrt(squared_norm)

    scale = min(
        1.0,
        clip_norm / (update_norm.item() + 1e-12)
    )

    protected = {}
    for key, delta in deltas.items():
        clipped_delta = delta * scale
        noise = torch.randn_like(clipped_delta) * (
            noise_multiplier * clip_norm
        )
        protected[key] = clipped_delta + noise

    return protected, update_norm.item(), scale

protected_updates = {}
privacy_rows = []

for name, state in retrained_states.items():
    update, original_norm, clip_

In [10]:

import json
from pathlib import Path

path = Path(
    "D:/adaptive_federated_gnn/notebooks/day17_privacy_mechanism.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))

keywords = [
    "def protect_client_update",
    "protected_updates =",
    "privacy_df =",
    "privacy_update_diagnostics.csv",
]

for i, cell in enumerate(nb["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(term in source for term in keywords):
        print(f"\n--- Day 17, Cell {i} ---")
        print(source)



--- Day 17, Cell 6 ---
CLIP_NORM = 1.0
NOISE_MULTIPLIER = 0.5

def protect_client_update(local_state, base_state,
                          clip_norm=1.0, noise_multiplier=0.5):
    # Compute one global L2 norm over all parameter updates.
    deltas = {
        key: local_state[key].float() - base_state[key].float()
        for key in base_state
    }

    squared_norm = sum(
        torch.sum(delta ** 2) for delta in deltas.values()
    )
    update_norm = torch.sqrt(squared_norm)

    scale = min(
        1.0,
        clip_norm / (update_norm.item() + 1e-12)
    )

    protected = {}
    for key, delta in deltas.items():
        clipped_delta = delta * scale
        noise = torch.randn_like(clipped_delta) * (
            noise_multiplier * clip_norm
        )
        protected[key] = clipped_delta + noise

    return protected, update_norm.item(), scale

protected_updates = {}
privacy_rows = []

for name, state in retrained_states.items():
    update, original_norm, clip_scale = pro

In [11]:

import json
from pathlib import Path

path = Path(
    "D:/adaptive_federated_gnn/notebooks/day17_privacy_mechanism.ipynb"
)

nb = json.loads(path.read_text(encoding="utf-8"))
source = "".join(nb["cells"][5].get("source", []))

print("Total characters:", len(source))

for start in range(0, len(source), 1800):
    end = min(start + 1800, len(source))
    print(f"\n--- Characters {start} to {end} ---\n")
    print(source[start:end])


Total characters: 1578

--- Characters 0 to 1578 ---

CLIP_NORM = 1.0
NOISE_MULTIPLIER = 0.5

def protect_client_update(local_state, base_state,
                          clip_norm=1.0, noise_multiplier=0.5):
    # Compute one global L2 norm over all parameter updates.
    deltas = {
        key: local_state[key].float() - base_state[key].float()
        for key in base_state
    }

    squared_norm = sum(
        torch.sum(delta ** 2) for delta in deltas.values()
    )
    update_norm = torch.sqrt(squared_norm)

    scale = min(
        1.0,
        clip_norm / (update_norm.item() + 1e-12)
    )

    protected = {}
    for key, delta in deltas.items():
        clipped_delta = delta * scale
        noise = torch.randn_like(clipped_delta) * (
            noise_multiplier * clip_norm
        )
        protected[key] = clipped_delta + noise

    return protected, update_norm.item(), scale

protected_updates = {}
privacy_rows = []

for name, state in retrained_states.items():
    update, o

In [12]:

from pathlib import Path
import torch
import pandas as pd

BASE = Path("D:/adaptive_federated_gnn")
DAY17 = BASE / "outputs/day17"
OUT = BASE / "outputs/day24"
OUT.mkdir(parents=True, exist_ok=True)

base_state = torch.load(
    DAY17 / "shared_initial_model.pt",
    map_location="cpu",
    weights_only=True
)

protected_updates = torch.load(
    DAY17 / "protected_client_updates.pt",
    map_location="cpu",
    weights_only=True
)

rows = []

for client, updates in protected_updates.items():
    original_state = torch.load(
        DAY17 / f"{client}_retrained_model.pt",
        map_location="cpu",
        weights_only=True
    )

    original_sq = 0.0
    protected_sq = 0.0
    noise_sq = 0.0

    for key in base_state:
        base = base_state[key].float()
        original_delta = original_state[key].float() - base
        protected_delta = updates[key].float()

        estimated_noise = protected_delta - original_delta

        original_sq += original_delta.pow(2).sum().item()
        protected_sq += protected_delta.pow(2).sum().item()
        noise_sq += estimated_noise.pow(2).sum().item()

    rows.append({
        "Client": client,
        "Original Update Norm": original_sq ** 0.5,
        "Protected Update Norm": protected_sq ** 0.5,
        "Estimated Noise Norm": noise_sq ** 0.5
    })

diagnostics = pd.DataFrame(rows)
display(diagnostics)

save_path = OUT / "noise_impact_diagnostics.csv"
diagnostics.to_csv(save_path, index=False)

print("Saved:", save_path)


,Client,Original Update Norm,Protected Update Norm,Estimated Noise Norm
0,client_1,0.400369,142.541508,142.538848
1,client_2,0.305792,142.064084,142.065085
2,client_3,0.465926,142.295940,142.298981


Saved: D:\adaptive_federated_gnn\outputs\day24\noise_impact_diagnostics.csv


In [13]:

from pathlib import Path

BASE = Path("D:/adaptive_federated_gnn")
DAY17 = BASE / "outputs/day17"
DAY24 = BASE / "outputs/day24"

DAY24.mkdir(parents=True, exist_ok=True)

required_files = [
    "shared_initial_model.pt",
    "protected_client_updates.pt",
    "client_1_retrained_model.pt",
    "client_2_retrained_model.pt",
    "client_3_retrained_model.pt",
    "private_fedavg_weights.csv",
]

print("DAY 24 FILE CHECK")
print("-" * 40)

for filename in required_files:
    path = DAY17 / filename
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{filename}: {status}")

print("\nDay 24 output folder:", DAY24)


DAY 24 FILE CHECK
----------------------------------------
shared_initial_model.pt: FOUND
protected_client_updates.pt: FOUND
client_1_retrained_model.pt: FOUND
client_2_retrained_model.pt: FOUND
client_3_retrained_model.pt: FOUND
private_fedavg_weights.csv: FOUND

Day 24 output folder: D:\adaptive_federated_gnn\outputs\day24


In [14]:

# DAY 24: Compare no-noise, reduced-noise, and current private models

from pathlib import Path
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Paths
BASE = Path("D:/adaptive_federated_gnn")
DAY10 = BASE / "outputs/day10"
DAY16 = BASE / "outputs/day16"
DAY17 = BASE / "outputs/day17"
DAY24 = BASE / "outputs/day24"
DAY24.mkdir(parents=True, exist_ok=True)

torch.manual_seed(2401)
np.random.seed(2401)

# 2. Load graph and saved client models
graph = torch.load(
    BASE / "data/processed/edge_graph.pt",
    map_location="cpu",
    weights_only=False
)

edge_index = graph["edge_index"].long()
feature_key = "edge_attr" if "edge_attr" in graph else "edge_features"
label_key = "edge_label" if "edge_label" in graph else "edge_labels"

edge_features = graph[feature_key].float()
edge_labels = graph[label_key].long().view(-1)

num_nodes = int(edge_index.max().item()) + 1
feature_size = edge_features.shape[1]

shared_state = torch.load(
    DAY17 / "shared_initial_model.pt",
    map_location="cpu",
    weights_only=True
)

client_states = {}
for name in ["client_1", "client_2", "client_3"]:
    client_states[name] = torch.load(
        DAY17 / f"{name}_retrained_model.pt",
        map_location="cpu",
        weights_only=True
    )

# 3. Define the same EdgeGCN architecture used in Day 17
class EdgeGCN(nn.Module):
    def __init__(self, num_nodes, edge_feature_size,
                 embedding_size=16, hidden_size=32):
        super().__init__()
        self.node_embeddings = nn.Embedding(
            num_nodes, embedding_size
        )
        self.conv1 = GCNConv(embedding_size, hidden_size)
        self.conv2 = GCNConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + edge_feature_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, edge_index, edge_features):
        node_ids = torch.arange(
            self.node_embeddings.num_embeddings,
            device=edge_index.device
        )
        x = self.node_embeddings(node_ids)
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        source = x[edge_index[0]]
        destination = x[edge_index[1]]
        combined = torch.cat(
            [source, destination, edge_features], dim=1
        )
        return self.classifier(combined)

# 4. Recover the same held-out test edges used in Day 17
day10_clients = torch.load(
    DAY10 / "federated_clients.pt",
    map_location="cpu",
    weights_only=False
)

train_indices = np.unique(np.concatenate([
    data["local_edge_indices"].cpu().numpy()
    if torch.is_tensor(data["local_edge_indices"])
    else np.asarray(data["local_edge_indices"])
    for data in day10_clients.values()
])).astype(np.int64)

test_indices = np.setdiff1d(
    np.arange(len(edge_labels)), train_indices
)

if len(test_indices) == 0:
    raise ValueError("No held-out test edges were found.")

test_idx = torch.tensor(test_indices, dtype=torch.long)
test_y = edge_labels[test_idx].numpy()

# 5. Read the original aggregation weights
weights_df = pd.read_csv(DAY17 / "private_fedavg_weights.csv")
print("Aggregation weight columns:", weights_df.columns.tolist())

weights = dict(zip(
    weights_df["Client"].astype(str),
    weights_df["Aggregation Weight"].astype(float)
))

# 6. Build FedAvg updates with a chosen noise level
def build_global_model(noise_multiplier):
    global_state = {}

    for key, base_tensor in shared_state.items():
        combined_delta = torch.zeros_like(
            base_tensor, dtype=torch.float32
        )

        for name, local_state in client_states.items():
            delta = (
                local_state[key].float()
                - shared_state[key].float()
            )

            # Reproduce Day 17's global update clipping
            # independently for each client.
            total_sq = sum(
                (
                    client_states[name][k].float()
                    - shared_state[k].float()
                ).pow(2).sum()
                for k in shared_state
            )
            update_norm = torch.sqrt(total_sq)
            scale = min(1.0, 1.0 / (update_norm.item() + 1e-12))
            clipped_delta = delta * scale

            # Noise multiplier 0.0 means no added noise.
            noise = torch.randn_like(clipped_delta) * noise_multiplier
            protected_delta = clipped_delta + noise

            combined_delta += weights[name] * protected_delta

        global_state[key] = (
            shared_state[key].float() + combined_delta
        )

    model = EdgeGCN(num_nodes, feature_size)
    model.load_state_dict(global_state)
    model.eval()
    return model

# 7. Evaluate a model on the same test edges
def evaluate_model(model, model_name):
    with torch.no_grad():
        logits = model(edge_index, edge_features)[test_idx]
        predictions = logits.argmax(dim=1).numpy()

    tn, fp, fn, tp = confusion_matrix(
        test_y, predictions, labels=[0, 1]
    ).ravel()

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(test_y, predictions),
        "Precision": precision_score(
            test_y, predictions, zero_division=0
        ),
        "Recall": recall_score(
            test_y, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            test_y, predictions, zero_division=0
        ),
        "False Positive Rate": fp / (fp + tn) if (fp + tn) else 0,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

# 8. Test no noise and reduced noise
no_noise_model = build_global_model(0.0)
reduced_noise_model = build_global_model(0.05)

results = [
    evaluate_model(no_noise_model, "FedAvg - No Noise"),
    evaluate_model(reduced_noise_model, "FedAvg - Noise 0.05")
]

# Load the already saved Day 17 private model
current_private_model = EdgeGCN(num_nodes, feature_size)
current_private_model.load_state_dict(torch.load(
    DAY17 / "private_global_model.pt",
    map_location="cpu",
    weights_only=True
))
current_private_model.eval()

results.append(
    evaluate_model(current_private_model, "Day 17 - Existing Private Model")
)

results_df = pd.DataFrame(results)
display(results_df.round(4))

save_path = DAY24 / "noise_level_comparison.csv"
results_df.to_csv(save_path, index=False)

# Save the experimental models too
torch.save(
    no_noise_model.state_dict(),
    DAY24 / "fedavg_no_noise_model.pt"
)
torch.save(
    reduced_noise_model.state_dict(),
    DAY24 / "fedavg_noise_005_model.pt"
)

print("\nComparison saved to:", save_path)
print("Models saved in:", DAY24)
print("Test samples:", len(test_y))
print("Actual test label counts:")
print(pd.Series(test_y).value_counts().sort_index())


c:\Users\daksh\anaconda3\envs\adaptive_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Aggregation weight columns: ['Client', 'Training Edges', 'Aggregation Weight']


,Model,Accuracy,Precision,Recall,F1 Score,False Positive Rate,TN,FP,FN,TP
0,FedAvg - No Noise,0.4133,0.0080,0.0005,0.0009,0.0717,35449,2739,47619,22
1,FedAvg - Noise 0.05,0.8222,0.7779,0.9513,0.8559,0.3389,25245,12943,2318,45323
2,Day 17 - Existing Private Model,0.5728,0.5651,0.9995,0.7220,0.9595,1546,36642,25,47616



Comparison saved to: D:\adaptive_federated_gnn\outputs\day24\noise_level_comparison.csv
Models saved in: D:\adaptive_federated_gnn\outputs\day24
Test samples: 85829
Actual test label counts:
0    38188
1    47641
Name: count, dtype: int64


In [15]:

# DAY 24: Check stability across five random noise samples

import pandas as pd
import torch
from pathlib import Path

DAY24 = Path("D:/adaptive_federated_gnn/outputs/day24")

rows = []

for seed in [11, 22, 33, 44, 55]:
    torch.manual_seed(seed)

    model = build_global_model(0.05)
    result = evaluate_model(model, f"Noise 0.05, seed {seed}")
    result["Seed"] = seed
    rows.append(result)

stability_df = pd.DataFrame(rows)

display(
    stability_df[
        [
            "Seed",
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "False Positive Rate"
        ]
    ].round(4)
)

metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "False Positive Rate"
]

print("\nAverage across five runs:")
display(stability_df[metrics].mean().to_frame("Mean").round(4))

print("\nStandard deviation across five runs:")
display(stability_df[metrics].std().to_frame("Std Dev").round(4))

save_path = DAY24 / "noise_005_stability_test.csv"
stability_df.to_csv(save_path, index=False)

print("Saved:", save_path)


,Seed,Accuracy,Precision,Recall,F1 Score,False Positive Rate
0,11,0.6375,0.7670,0.4982,0.6041,0.1888
1,22,0.8364,0.8061,0.9287,0.8631,0.2787
2,33,0.4109,0.0081,0.0005,0.0009,0.0772
3,44,0.6927,0.9244,0.4861,0.6372,0.0496
4,55,0.3362,0.0081,0.0016,0.0027,0.2464



Average across five runs:


,Mean
Accuracy,0.5827
Precision,0.5027
Recall,0.3830
F1 Score,0.4216
False Positive Rate,0.1681



Standard deviation across five runs:


,Std Dev
Accuracy,0.2060
Precision,0.4553
Recall,0.3916
F1 Score,0.3959
False Positive Rate,0.1014


Saved: D:\adaptive_federated_gnn\outputs\day24\noise_005_stability_test.csv


In [16]:

# DAY 24: Compare noise levels across five random seeds

import pandas as pd
import torch
from pathlib import Path

DAY24 = Path("D:/adaptive_federated_gnn/outputs/day24")
DAY24.mkdir(parents=True, exist_ok=True)

noise_levels = [0.001, 0.005, 0.01, 0.05]
seeds = [11, 22, 33, 44, 55]

rows = []

for noise_level in noise_levels:
    for seed in seeds:
        torch.manual_seed(seed)

        model = build_global_model(noise_level)
        result = evaluate_model(
            model,
            f"Noise {noise_level}, seed {seed}"
        )

        result["Noise Multiplier"] = noise_level
        result["Seed"] = seed
        rows.append(result)

        print(
            f"Noise={noise_level}, seed={seed}, "
            f"F1={result['F1 Score']:.4f}, "
            f"accuracy={result['Accuracy']:.4f}"
        )

results_df = pd.DataFrame(rows)

raw_path = DAY24 / "noise_grid_all_runs.csv"
results_df.to_csv(raw_path, index=False)

summary_df = (
    results_df
    .groupby("Noise Multiplier")[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "False Positive Rate"
        ]
    ]
    .agg(["mean", "std"])
    .round(4)
)

display(summary_df)

summary_path = DAY24 / "noise_grid_summary.csv"
summary_df.to_csv(summary_path)

print("\nAll runs saved:", raw_path)
print("Summary saved:", summary_path)


Noise=0.001, seed=11, F1=0.0009, accuracy=0.4125
Noise=0.001, seed=22, F1=0.0009, accuracy=0.4128
Noise=0.001, seed=33, F1=0.0009, accuracy=0.4130
Noise=0.001, seed=44, F1=0.0009, accuracy=0.4143
Noise=0.001, seed=55, F1=0.0009, accuracy=0.4128
Noise=0.005, seed=11, F1=0.0009, accuracy=0.4087
Noise=0.005, seed=22, F1=0.0009, accuracy=0.4110
Noise=0.005, seed=33, F1=0.0009, accuracy=0.4119
Noise=0.005, seed=44, F1=0.0006, accuracy=0.4178
Noise=0.005, seed=55, F1=0.0009, accuracy=0.4105
Noise=0.01, seed=11, F1=0.0009, accuracy=0.4035
Noise=0.01, seed=22, F1=0.0034, accuracy=0.4095
Noise=0.01, seed=33, F1=0.0009, accuracy=0.4106
Noise=0.01, seed=44, F1=0.0003, accuracy=0.4209
Noise=0.01, seed=55, F1=0.0009, accuracy=0.4070
Noise=0.05, seed=11, F1=0.6041, accuracy=0.6375
Noise=0.05, seed=22, F1=0.8631, accuracy=0.8364
Noise=0.05, seed=33, F1=0.0009, accuracy=0.4109
Noise=0.05, seed=44, F1=0.6372, accuracy=0.6927
Noise=0.05, seed=55, F1=0.0027, accuracy=0.3362


Accuracy         Precision          Recall         F1 Score  \
                     mean     std      mean     std    mean     std     mean   
Noise Multiplier                                                               
0.001              0.4131  0.0007    0.0079  0.0002  0.0005  0.0000   0.0009   
0.005              0.4120  0.0034    0.0073  0.0008  0.0004  0.0001   0.0008   
0.010              0.4103  0.0065    0.0105  0.0094  0.0007  0.0006   0.0013   
0.050              0.5827  0.2060    0.5027  0.4553  0.3830  0.3916   0.4216   

                         False Positive Rate          
                     std                mean     std  
Noise Multiplier                                      
0.001             0.0000              0.0721  0.0016  
0.005             0.0002              0.0746  0.0078  
0.010             0.0012              0.0787  0.0149  
0.050             0.3959              0.1681  0.1014


All runs saved: D:\adaptive_federated_gnn\outputs\day24\noise_grid_all_runs.csv
Summary saved: D:\adaptive_federated_gnn\outputs\day24\noise_grid_summary.csv


In [19]:
# Check the exact metric names returned by evaluate_model()
torch.manual_seed(11)

test_model = build_global_model(0.02)
test_metrics = evaluate_model(test_model, "metric_key_test")

print("Available metric keys:")
print(list(test_metrics.keys()))

print("\nFull metrics:")
print(test_metrics)

Available metric keys:
['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'False Positive Rate', 'TN', 'FP', 'FN', 'TP']

Full metrics:
{'Model': 'metric_key_test', 'Accuracy': 0.39208193034988176, 'Precision': 0.005451373746184038, 'Recall': 0.0005247580865221133, 'F1 Score': 0.000957359220326651, 'False Positive Rate': np.float64(0.11943542474075626), 'TN': 33627, 'FP': 4561, 'FN': 47616, 'TP': 25}


In [ ]:

import pandas as pd
import torch
from pathlib import Path

noise_levels = [0.015, 0.020, 0.025, 0.030, 0.035, 0.040, 0.045]
seeds = [11, 22, 33, 44, 55]

results = []

for noise_level in noise_levels:
    for seed in seeds:
        torch.manual_seed(seed)

        model = build_global_model(noise_level)
        metrics = evaluate_model(model, f"noise_{noise_level}_seed_{seed}")

        results.append({
            "noise_multiplier": noise_level,
            "seed": seed,
            "accuracy": metrics["Accuracy"],
            "precision": metrics["Precision"],
            "recall": metrics["Recall"],
            "f1_score": metrics["F1 Score"],
            "false_positive_rate": metrics["False Positive Rate"]
        })

        print(
            f"Noise={noise_level:.3f}, seed={seed}, "
            f"F1={metrics['']:.4f}, "
            f"accuracy={metrics['Accuracy']:.4f}"
        )

results_df = pd.DataFrame(results)

output_dir = Path("D:/adaptive_federated_gnn/outputs/day24")
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "intermediate_noise_all_runs.csv",
    index=False
)

summary_df = results_df.groupby("noise_multiplier").agg(
    {
        "accuracy": ["mean", "std"],
        "precision": ["mean", "std"],
        "recall": ["mean", "std"],
        "f1_score": ["mean", "std"],
        "false_positive_rate": ["mean", "std"]
    }
).round(4)

summary_df.to_csv(output_dir / "intermediate_noise_summary.csv")

display(summary_df)


Noise=0.015, seed=11, F1=0.0010, accuracy=0.3984
Noise=0.015, seed=22, F1=0.5648, accuracy=0.6381
Noise=0.015, seed=33, F1=0.0009, accuracy=0.4096
Noise=0.015, seed=44, F1=0.0005, accuracy=0.4232
Noise=0.015, seed=55, F1=0.0017, accuracy=0.4004
Noise=0.020, seed=11, F1=0.0010, accuracy=0.3921
Noise=0.020, seed=22, F1=0.5873, accuracy=0.6488
Noise=0.020, seed=33, F1=0.0009, accuracy=0.4083
Noise=0.020, seed=44, F1=0.5984, accuracy=0.6704
Noise=0.020, seed=55, F1=0.0018, accuracy=0.3911
Noise=0.025, seed=11, F1=0.0009, accuracy=0.3853
Noise=0.025, seed=22, F1=0.5952, accuracy=0.6499
Noise=0.025, seed=33, F1=0.0009, accuracy=0.4080
Noise=0.025, seed=44, F1=0.6252, accuracy=0.6875
Noise=0.025, seed=55, F1=0.0017, accuracy=0.3839
Noise=0.030, seed=11, F1=0.0082, accuracy=0.3824
Noise=0.030, seed=22, F1=0.5992, accuracy=0.6477
Noise=0.030, seed=33, F1=0.0009, accuracy=0.4086
Noise=0.030, seed=44, F1=0.6358, accuracy=0.6949
Noise=0.030, seed=55, F1=0.0018, accuracy=0.3761
Noise=0.035, seed=11

accuracy         precision          recall         f1_score  \
                     mean     std      mean     std    mean     std     mean   
noise_multiplier                                                               
0.015              0.4540  0.1034    0.1762  0.3763  0.0851  0.1890   0.1138   
0.020              0.5021  0.1441    0.3583  0.4811  0.1789  0.2441   0.2379   
0.025              0.5029  0.1522    0.3573  0.4811  0.1871  0.2553   0.2448   
0.030              0.5020  0.1560    0.3617  0.4734  0.1921  0.2602   0.2492   
0.035              0.5383  0.1481    0.4966  0.4537  0.2717  0.2535   0.3495   
0.040              0.5673  0.1790    0.5087  0.4633  0.3339  0.3241   0.3961   
0.045              0.5827  0.2027    0.5069  0.4602  0.3738  0.3811   0.4180   

                         false_positive_rate          
                     std                mean     std  
noise_multiplier                                      
0.015             0.2521              0.0858  0.0227  
0.020             0.3241              0.0946  0.0318  
0.025             0.3337              0.1031  0.0410  
0.030             0.3365              0.1114  0.0506  
0.035             0.3231              0.1292  0.0691  
0.040             0.3679              0.1416  0.0837  
0.045             0.3932              0.1568  0.0929

In [23]:

import pandas as pd
from pathlib import Path

results_path = Path(
    "D:/adaptive_federated_gnn/outputs/day24/intermediate_noise_all_runs.csv"
)

runs = pd.read_csv(results_path)

# Show the best and worst runs
print("TOP 10 RUNS BY F1 SCORE")
display(
    runs.sort_values("f1_score", ascending=False)
        [["noise_multiplier", "seed", "accuracy",
          "precision", "recall", "f1_score",
          "false_positive_rate"]]
        .head(10)
)

print("LOWEST 10 RUNS BY F1 SCORE")
display(
    runs.sort_values("f1_score", ascending=True)
        [["noise_multiplier", "seed", "accuracy",
          "precision", "recall", "f1_score",
          "false_positive_rate"]]
        .head(10)
)

print("PREDICTION QUALITY BY SEED")
display(
    runs.groupby("seed").agg(
        mean_f1=("f1_score", "mean"),
        std_f1=("f1_score", "std"),
        mean_recall=("recall", "mean"),
        mean_fpr=("false_positive_rate", "mean")
    ).round(4)
)


TOP 10 RUNS BY F1 SCORE


,noise_multiplier,seed,accuracy,precision,recall,f1_score,false_positive_rate
31,0.045,22,0.832597,0.815833,0.902038,0.856773,0.254033
26,0.040,22,0.757658,0.808865,0.737726,0.771659,0.217477
28,0.040,44,0.700311,0.953115,0.483890,0.641895,0.029695
23,0.035,44,0.698249,0.948141,0.482777,0.639786,0.032942
33,0.045,44,0.695709,0.935885,0.485023,0.638924,0.041453
18,0.030,44,0.694917,0.942173,0.479818,0.635829,0.036739
13,0.025,44,0.687483,0.935050,0.469596,0.625204,0.040693
21,0.035,22,0.648173,0.779336,0.510779,0.617105,0.180423
16,0.030,22,0.647730,0.813125,0.474381,0.599191,0.136011
8,0.020,44,0.670426,0.924430,0.442413,0.598430,0.045119


LOWEST 10 RUNS BY F1 SCORE


,noise_multiplier,seed,accuracy,precision,recall,f1_score,false_positive_rate
3,0.015,44,0.423237,0.006363,0.000252,0.000485,0.049073
12,0.025,33,0.408021,0.007463,0.000504,0.000944,0.083586
7,0.020,33,0.408254,0.007509,0.000504,0.000944,0.083063
17,0.030,33,0.408638,0.007588,0.000504,0.000945,0.082199
22,0.035,33,0.408999,0.007663,0.000504,0.000945,0.081387
2,0.015,33,0.409570,0.007785,0.000504,0.000946,0.080104
10,0.025,11,0.385324,0.004839,0.000525,0.000947,0.134623
27,0.040,33,0.409931,0.007864,0.000504,0.000947,0.079292
32,0.045,33,0.410001,0.007879,0.000504,0.000947,0.079135
5,0.020,11,0.392082,0.005451,0.000525,0.000957,0.119435


PREDICTION QUALITY BY SEED


,mean_f1,std_f1,mean_recall,mean_fpr
seed,,,,
11,0.2364,0.2930,0.1852,0.1446
22,0.6560,0.1119,0.5660,0.1576
33,0.0009,0.0000,0.0005,0.0813
44,0.5401,0.2384,0.4063,0.0394
55,0.0018,0.0001,0.0010,0.1646


In [24]:

import torch
import pandas as pd
from pathlib import Path

base_path = Path("D:/adaptive_federated_gnn/outputs/day17")

shared_state = torch.load(
    base_path / "shared_initial_model.pt",
    map_location="cpu",
    weights_only=True
)

client_names = ["client_1", "client_2", "client_3"]
client_states = {}

for name in client_names:
    path = base_path / f"{name}_retrained_model.pt"
    client_states[name] = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

rows = []

for name, state in client_states.items():
    squared_norm = 0.0
    squared_base_norm = 0.0
    dot_product = 0.0

    for key, base_tensor in shared_state.items():
        delta = state[key].float() - base_tensor.float()
        base = base_tensor.float()

        squared_norm += delta.pow(2).sum().item()
        squared_base_norm += base.pow(2).sum().item()
        dot_product += (delta * base).sum().item()

    rows.append({
        "client": name,
        "update_norm": squared_norm ** 0.5,
        "base_model_norm": squared_base_norm ** 0.5,
        "update_to_base_ratio": (
            squared_norm ** 0.5 /
            (squared_base_norm ** 0.5 + 1e-12)
        ),
        "update_base_cosine": (
            dot_product /
            ((squared_norm ** 0.5) *
             (squared_base_norm ** 0.5) + 1e-12)
        )
    })

diagnostics = pd.DataFrame(rows)
display(diagnostics.round(6))


,client,update_norm,base_model_norm,update_to_base_ratio,update_base_cosine
0,client_1,0.400369,266.473847,0.001502,-0.004863
1,client_2,0.305792,266.473847,0.001148,-0.008918
2,client_3,0.465926,266.473847,0.001748,-0.003463


In [25]:

import torch
import pandas as pd
from itertools import combinations

# Use shared_state and client_states from the previous cell.
updates = {}

for name, state in client_states.items():
    updates[name] = torch.cat([
        (state[key].float() - shared_state[key].float()).reshape(-1)
        for key in shared_state
    ])

rows = []

for a, b in combinations(updates.keys(), 2):
    u = updates[a]
    v = updates[b]

    cosine = torch.dot(u, v) / (
        torch.linalg.vector_norm(u) *
        torch.linalg.vector_norm(v) + 1e-12
    )

    rows.append({
        "client_pair": f"{a} vs {b}",
        "cosine_similarity": cosine.item(),
        "update_norm_a": torch.linalg.vector_norm(u).item(),
        "update_norm_b": torch.linalg.vector_norm(v).item()
    })

display(pd.DataFrame(rows).round(6))


,client_pair,cosine_similarity,update_norm_a,update_norm_b
0,client_1 vs client_2,0.345872,0.400369,0.305792
1,client_1 vs client_3,0.305243,0.400369,0.465926
2,client_2 vs client_3,0.294106,0.305792,0.465926


In [26]:

# Check whether the current no-noise and noisy results are already saved.

from pathlib import Path
import pandas as pd

output_dir = Path("D:/adaptive_federated_gnn/outputs/day24")

for filename in [
    "noise_level_comparison.csv",
    "intermediate_noise_all_runs.csv",
    "intermediate_noise_summary.csv"
]:
    path = output_dir / filename
    print(f"\n{filename}: {'FOUND' if path.exists() else 'NOT FOUND'}")

    if path.exists():
        df = pd.read_csv(path)
        print(df.head(5).to_string(index=False))



noise_level_comparison.csv: FOUND
                          Model  Accuracy  Precision   Recall  F1 Score  False Positive Rate    TN    FP    FN    TP
              FedAvg - No Noise  0.413275   0.007968 0.000462  0.000873             0.071724 35449  2739 47619    22
            FedAvg - Noise 0.05  0.822193   0.777864 0.951344  0.855902             0.338928 25245 12943  2318 45323
Day 17 - Existing Private Model  0.572790   0.565121 0.999475  0.722007             0.959516  1546 36642    25 47616

intermediate_noise_all_runs.csv: FOUND
 noise_multiplier  seed  accuracy  precision   recall  f1_score  false_positive_rate
            0.015    11  0.398443   0.006188 0.000525  0.000967             0.105138
            0.015    22  0.638106   0.849229 0.423144  0.564844             0.093721
            0.015    33  0.409570   0.007785 0.000504  0.000946             0.080104
            0.015    44  0.423237   0.006363 0.000252  0.000485             0.049073
            0.015    55  0.40041

In [27]:

from pathlib import Path
import pandas as pd

weights_path = Path(
    "D:/adaptive_federated_gnn/outputs/day17/private_fedavg_weights.csv"
)

weights_df = pd.read_csv(weights_path)

print("Saved FedAvg weights:")
display(weights_df)

print("\nColumn names:")
print(weights_df.columns.tolist())

print("\nSum of weights:")
numeric_cols = weights_df.select_dtypes(include="number").columns
for col in numeric_cols:
    if "weight" in col.lower():
        print(f"{col}: {weights_df[col].sum():.6f}")


Saved FedAvg weights:


,Client,Training Edges,Aggregation Weight
0,client_1,66756,0.333337
1,client_2,66755,0.333332
2,client_3,66755,0.333332



Column names:
['Client', 'Training Edges', 'Aggregation Weight']

Sum of weights:
Aggregation Weight: 1.000000


In [28]:

import torch
import pandas as pd

# Compare the loaded client checkpoints with the shared starting model.
rows = []

for name, state in client_states.items():
    total_squared_difference = 0.0
    total_parameters = 0
    mismatched_keys = []

    if set(state.keys()) != set(shared_state.keys()):
        mismatched_keys = list(
            set(state.keys()).symmetric_difference(shared_state.keys())
        )

    for key in shared_state:
        if key not in state:
            continue

        if state[key].shape != shared_state[key].shape:
            mismatched_keys.append(key)
            continue

        difference = (
            state[key].float() - shared_state[key].float()
        )

        total_squared_difference += difference.pow(2).sum().item()
        total_parameters += difference.numel()

    rows.append({
        "client": name,
        "parameter_count_compared": total_parameters,
        "update_norm": total_squared_difference ** 0.5,
        "mismatched_keys": len(set(mismatched_keys))
    })

display(pd.DataFrame(rows).round(6))


,client,parameter_count_compared,update_norm,mismatched_keys
0,client_1,80642,0.400369,0
1,client_2,80642,0.305792,0
2,client_3,80642,0.465926,0
